# Example: Virtual circuit calculations

---

This example notebook will demonstrate how to construct **virtual circuits** (VCs) for a **MAST-U-like tokamak** equilibrium using JAX.

### What are Virtual Circuits and why are they useful?

VCs can be used to identify which (active) poloidal field (PF) coils have the most significant impact on a set of specified plasma shape parameters (which we will refer to henceforth as **targets**). The targets are equilibrium-related quantities such as the inner or outer midplane radii $R_{in}$ or $R_{out}$ (more will be listed later on). 

More formally, the **virtual circuit** (VC) matrix $V$ for a given equilibrium (and chosen set of shape targets and PF coils) is defined as

$$ V = (S^T S)^{-1} S^T, $$

where $S$ is the **shape** (Jacobian) matrix:

$$ S_{i,j} = \frac{\partial T_i}{\partial I_j}. $$

Here, $T_i$ is the $i$ th target and $I_j$ is the current in the $j$ th PF coil. We note that $V$ is simply the Moore-Penrose pseudo-inverse of $S$. In this example, $S$ will be calculated using JAX. 

### How can these VCs be used?

Once we know the VC matrix $V$ for a given equilibrium (and its associated target values $\vec{T}$), we can specify a perturbation in the targets $\vec{\Delta T}$ and calculate the change in coil currents required to acheive the new targets. The shifts in coil currents can be found via:

$$ \vec{\Delta I} = V \vec{\Delta T}. $$

Using $\vec{\Delta I}$ we can perturb the coil currents in our equilibrium, re-solve the static forward Grad-Shafranov (GS) problem, and observe how the targets (call them $\vec{T}_{new}$) have changed 
in the new equilibrium vs. the old targets $\vec{T}$.

 ---


### Generate a starting equilibrums

Firstly, we need an equilbirium to test the VCs on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
    magnetic_probe_path=f"../machine_configs/MAST-U/MAST-U_like_magnetic_probes.pickle",
)

# initialise the equilibrium
from freegsnke import equilibrium_update
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # Radial range
    Zmin=-2.2, Zmax=2.2,  # Vertical range
    nx=65,                # Number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # Number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)  

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# initialise the limiter geometry
from freegsnke.jaxify import limiter_func
jLimiter = limiter_func.Limiter_handler(eq, eq.tokamak.limiter)

# initialise the profiles
from freegsnke.jaxify.jtor import JConstrainPaxisIp, JLao85
jProfile = JConstrainPaxisIp(
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

# initialise the static solver
from freegsnke.jaxify import GSstaticsolver
jGS = GSstaticsolver.NKGSsolver(eq, jProfile, jLimiter)
jProfilePars = jProfile.init_params

# set the coil currents
import pickle
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)
for key in current_values.keys():
    eq.tokamak.set_coil_current(key, current_values[key])

eq.tokamak.getCurrents()
jCurrVec=jnp.asarray(eq.tokamak.getCurrentsVec())

# carry out the foward solve to find the equilibrium
psi = jGS.solve(init_psi=eq.psi(),
                profilePars = jProfilePars,
                currentvec = jCurrVec,
                target_relative_tolerance = 1e-4,
                verbose=True, # print output
                )

# plot the resulting equilbria 
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
plt.contour(eq.R,eq.Z,psi,20)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Define shape targets

Next we need to define the shape targets (i.e. our quantities of interest) that we wish to monitor. FreeGSNKE has a number of pre-defined targets:
- "R_in": inner midplane radius.  
- "R_out": outer midplane radius.  
- "Rx_lower": lower X-point (radial) position.
- "Zx_lower": lower X-point (vertical) position.
- "Rx_upper": upper X-point (radial) position.
- "Zx_upper": upper X-point (vertical) position.
- "Rs_lower_outer": lower strikepoint (radial) position.
- "Rs_upper_outer": upper strikepoint (radial) position.

In order to evaluate their sensitivities using JAX, we need to write native JAX implementations calculating these quantities. Then, we would easily get their derivatives wrt coil currents using JAX's automatic differentiation capabilities.

In [ ]:

def targets(Ic, init_psi, targets_list):

	# Get full current vector from subset of control currents
	jCurr = Ic

	# Solve the forward GS problem using this set of coil currents
	psi = jGS.solve(init_psi=init_psi,
                    profilePars = jProfilePars,
                    currentvec = jCurr,
                    target_relative_tolerance = 1e-8,
                    verbose=False, # print output
                    )
	
	# Set up an interpolation to get the values of psi at the control points
	import interpax as ix
	r1d = jGS.R[:,0]
	Rmin = jnp.amin(r1d)
	Rmax = jnp.amax(r1d)

	z1d = jGS.Z[0,:]
	# psi_func = ix.Interpolator2D(r1d,z1d,psi)

	# Get x and o-points
	opts,xpts = jGS.critpoints(psi)
	psi_o = opts[0,2]
	psi_x = xpts[0,2]
	primary_x = xpts[0:2,:]

	diverted_mask = jGS.mask(psi, opts, xpts)

	dmask_inside_limiter = diverted_mask*jGS.limiter.mask_inside_limiter 
	psi_l, limiter_mask = jGS.limiter.core_mask_limiter(
								jGS.R, jGS.Z, 
								psi, psi_x, 
								dmask_inside_limiter, 
								jGS.limiter.limiter_mask_out)
	lmask_sum = jnp.sum(limiter_mask * jGS.limiter.mask_inside_limiter)
	# plasma_domain_mask = jnp.where(lmask_sum==0,
	# 					dmask_inside_limiter,
	# 					limiter_mask)
	psi_b = jnp.where(lmask_sum==0,psi_x,psi_l)

	# Find the closest index to requested Z
	Z = 0.0
	Zindex = jnp.argmin(abs(jGS.Z[0, :] - Z))

	# Normalise psi at this Z index
	psinorm = (psi[:, Zindex] - psi_o) / (
		psi_b - psi_o
	)

	# Start from the magnetic axis
	Rmagnetic = opts[0,0]
	Rindex_axis = jnp.argmin(abs(jGS.R[:, 0] - Rmagnetic))

	# Inner separatrix
	# Get the maximum index where psi > 1 in the R index range from 0 to Rindex_axis
	outside_inds = jnp.argwhere(psinorm[:Rindex_axis] > 1.0)

	if outside_inds.size == 0:
		R_in = Rmin
	else:
		Rindex_inner = jnp.amax(outside_inds)

		# Separatrix should now be between Rindex_inner and Rindex_inner+1
		# Linear interpolation
		R_in = (
			jGS.R[Rindex_inner, Zindex]
			* (1.0 - psinorm[Rindex_inner + 1])
			+ jGS.R[Rindex_inner + 1, Zindex]
			* (psinorm[Rindex_inner] - 1.0)
		) / (psinorm[Rindex_inner] - psinorm[Rindex_inner + 1])

	# Outer separatrix
	# Find the minimum index where psi > 1
	outside_inds = jnp.argwhere(psinorm[Rindex_axis:] > 1.0)

	if outside_inds.size == 0:
		R_out = Rmax
	else:
		Rindex_outer = jnp.amin(outside_inds) + Rindex_axis

		# Separatrix should now be between Rindex_outer-1 and Rindex_outer
		R_out = (
			jGS.R[Rindex_outer, Zindex]
			* (1.0 - psinorm[Rindex_outer - 1])
			+ jGS.R[Rindex_outer - 1, Zindex]
			* (psinorm[Rindex_outer] - 1.0)
		) / (psinorm[Rindex_outer] - psinorm[Rindex_outer - 1])

	i_lower = jnp.argmin(primary_x[:,1])
	Rx_lower = primary_x[i_lower,0]
	Zx_lower = primary_x[i_lower,1]

	i_upper = jnp.argmax(primary_x[:,1])
	Rx_upper = primary_x[i_upper,0]
	Zx_upper = primary_x[i_upper,1]

	target_vec = jnp.zeros(len(targets_list))

	# Create a mapping dictionary
	mapping = {
		"R_in": R_in,
		"R_out": R_out,
		"Rx_lower": Rx_lower,
		"Zx_lower": Zx_lower,
		"Rx_upper": Rx_upper,
		"Zx_upper": Zx_upper,
	}

	target_vec = jnp.array([mapping[target] for target in targets_list])

	return target_vec, (psi, psi_b, psi_o)

In [ ]:
# define the targets of interest
targets_list = ['R_in', 'R_out', 'Rx_lower', 'Zx_lower', 'Rx_upper', 'Zx_upper']

# calculate their values using the function defined above
target_values, state = targets(jCurrVec, psi, targets_list)

# print
for i in range(len(targets_list)):
    print(targets_list[i] + " = " + str(target_values[i]))


### Calculating the VCs

Now we've defined the targets we're interested in, we can begin calculating the **shape** and **virtual circuit** matrices. The following is a brief outline of how they're calculated:

##### 1. Initialise and solve the base equilibrium
We choose the coils that we would like to generate the virtual circuits for. This could be all the coils or a subset of the coils. This is passed as a list to the function.

##### 2. Find the shape matrix
Using the jax.value_and_grad function, we calculate the targets and the shape matrix together. The shape matrix gives the sensitivity of each of the targets $T_i$ with respect to a change in the $j$ th coil current.

If we want, we can also obtain the Jacobian matrix of the plasma current vector with respect to the coil currents: $\frac{\partial \vec{I}_y}{\partial \vec{I}}$.

##### 3. Find the virtual circuit matrix
Once the full shape matrix $S \in \Reals^{N_T \times N_c} $ is known, the **virtual circuit matrix** is computed as:

$$ V = (S^T S)^{-1} S^T \in \Reals^{N_c \times N_T}.$$

This matrix provides a mapping from requested shifts in the targets to the shifts in the coil currents required.

In [ ]:
# define which coils we wish to calculate the shape derivatives (and therefore VCs) for
coils = eq.tokamak.coils_list[0:12]
print(coils)

In [ ]:
# here we'll look at a subset of the coils and the targets
shaping_coils = ['D1', 'D2', 'D3', 'Dp', 'D5', 'D6', 'D7', 'P4', 'P5']
targets_list = ['R_in', 'R_out', 'Rx_lower', 'Zx_lower']

target_value, state = targets(jCurrVec, psi, targets_list)
Jac, state = jax.jacrev(targets,has_aux=True,argnums=0)(jCurrVec, psi, targets_list)


#  Create a mapping from label to index
index_map = {label: idx for idx, label in enumerate(eq.tokamak.coils_list)}

# Get indices for subset
indices = jnp.array([index_map[label] for label in shaping_coils])

shape_matrix = Jac[:,indices]

VC_matrix = jnp.linalg.pinv(shape_matrix)

### How do we make use of the VCs?

Now that we have the VCs, we can use them to idenitfy the coil current shifts required to change the targets by a certain amount. 

For example, we will ask for shifts in a few shape targets and observe what happens to the equilibrium once we apply the new currents from the VCs. 

In [ ]:
# let's set the requested shifts (units are metres) for the full set of targets
print(targets_list)
all_requested_target_shifts = [0.01, -0.01, 0.01, 0.01]

We specify a perturbation in the targets $\vec{\Delta T}$  (the shifts above) and calculate the change in coil currents required to achieve this by calculating:

$$ \vec{\Delta I} = V \vec{\Delta T}. $$

Using $\vec{\Delta I}$ we then perturb the coil currents in our original equilibrium, re-solve the static forward Grad-Shafranov (GS) problem, and return. This is all done in the following cell. 

In [ ]:
# we apply the VCs to the desired shifts to get the new shaping currents
delta_shaping_currs = VC_matrix @ jnp.array(all_requested_target_shifts)
old_shaping_currs = jCurrVec[indices]
new_shaping_currs = old_shaping_currs + delta_shaping_currs
print("Old shaping currents:", old_shaping_currs)
print("New shaping currents: ", new_shaping_currs)

# Get new full current vector
oldCurrVec = jCurrVec
newCurrVec = jCurrVec.at[indices].set(new_shaping_currs)

# Evaluate new targets
old_target_values, old_state = targets(oldCurrVec, psi, targets_list)
print(old_target_values)
new_target_values, new_state = targets(newCurrVec, psi, targets_list)
print(new_target_values)


We can then measure the accuracy of the VCs by observing the difference between the requested change in shape targets vs. those enacted by the VCs. 
 


In [ ]:
for i in range(len(targets_list)):
    print(f"Difference in {targets_list[i]} = {np.round(new_target_values[i] - old_target_values[i],3)} vs. requested = {(all_requested_target_shifts)[i]}.")


In the plots below, we can see the actual shifts by the VCs are almost exactly as those requested (for the targets with actual shifts). Note how the unshifted targets also shift under the VCs due to the nonlinear coupling between targets and coil currents. 

In [ ]:
# plots
rel_diff = np.abs(np.array(new_target_values) - np.array(old_target_values))/np.abs(old_target_values)

fig1, ((ax1), (ax2)) = plt.subplots(2, 1, figsize=(12,12), dpi=80)

ax1.grid(True, which='both', alpha=0.5)
ax1.scatter(targets_list, all_requested_target_shifts, color='red', marker='o', s=150, label="Requested")
ax1.scatter(targets_list, np.array(new_target_values) - np.array(old_target_values), color='royalblue', marker='o', s=75, label="Actual")
ax1.set_xlabel("Target")
ax1.set_ylabel("Shift [m]")
ax1.legend()
# ax1.set_ylim([-max(targets_shift+non_standard_targets_shift)*1.1, max(targets_shift+non_standard_targets_shift)*1.1])

ax2.grid(True, which='both', alpha=0.5)
ax2.scatter(targets_list, rel_diff, color='red', marker='o', s=150, edgecolors='black', label="Requested")
ax2.set_xlabel("Target")
ax2.set_ylabel("Relative shift")
labels = ax2.get_xticklabels()
ax2.set_yscale('log')
ax2.set_ylim([1e-6, 1e-0])

plt.tight_layout()

### Alternative VCs
We can then follow up by calculating alternative VCs for different target and coil combinations.

In [ ]:
# choose new coils and targets
new_shaping_coils = ['D1', 'D2', 'D3', 'Dp', 'D5', 'D6', 'D7', 'P4', 'P5']
new_targets_list = ['Rx_upper', 'Zx_upper'] # this time we use the upper targets

target_value, state = targets(jCurrVec, psi, new_targets_list)
Jac, state = jax.jacrev(targets,has_aux=True,argnums=0)(jCurrVec, psi, new_targets_list)

# Get indices for subset
indices = jnp.array([index_map[label] for label in new_shaping_coils])

shape_matrix = Jac[:,indices]

VC_matrix = jnp.linalg.pinv(shape_matrix)
print(VC_matrix.shape)

We can now apply this VC using the `apply_VC` method.

In [ ]:
all_requested_target_shifts=[0.02, -0.02]
delta_shaping_currs = VC_matrix @ jnp.array(all_requested_target_shifts)
old_shaping_currs = jCurrVec[indices]
new_shaping_currs = old_shaping_currs + delta_shaping_currs
print("Old shaping currents:", old_shaping_currs)
print("New shaping currents: ", new_shaping_currs)

# Get new full current vector
oldCurrVec = jCurrVec
newCurrVec = jCurrVec.at[indices].set(new_shaping_currs)

# Evaluate new targets
old_target_values, old_state = targets(oldCurrVec, psi, targets_list)
print(old_target_values)
new_target_values, new_state = targets(newCurrVec, psi, targets_list)
print(new_target_values)



In [ ]:
# plot the resulting equilbria 
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
ax1.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, color='k', linewidth=1.2, linestyle="-")
eq.tokamak.plot(axis=ax1, show=False)

ax1.contour(eq.R, eq.Z, old_state[0], levels=[old_state[1]], colors='r')
ax1.contour(eq.R, eq.Z, new_state[0], levels=[new_state[1]], colors='b', linestyles="--")

ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)

plt.tight_layout()

In [ ]:
# plots
rel_diff = np.abs(np.array(new_target_values) - np.array(old_target_values))/np.abs(old_target_values)

fig1, ((ax1), (ax2)) = plt.subplots(2, 1, figsize=(12,12), dpi=80)

ax1.grid(True, which='both', alpha=0.5)
ax1.scatter(target_names, all_requested_target_shifts, color='red', marker='o', s=150, label="Requested")
ax1.scatter(target_names, np.array(new_target_values) - np.array(old_target_values), color='royalblue', marker='o', s=75, label="Actual")
ax1.set_xlabel("Target")
ax1.set_ylabel("Shift [m]")
ax1.legend()
# ax1.set_ylim([-max(targets_shift+non_standard_targets_shift)*1.1, max(targets_shift+non_standard_targets_shift)*1.1])

ax2.grid(True, which='both', alpha=0.5)
ax2.scatter(target_names, rel_diff, color='red', marker='o', s=150, edgecolors='black', label="Requested")
ax2.set_xlabel("Target")
ax2.set_ylabel("Relative shift")
labels = ax2.get_xticklabels()
ax2.set_yscale('log')
ax2.set_ylim([1e-6, 1e-0])

plt.tight_layout()